**Utrecht Summer School 'Introduction to Complex Systems'**

Wednesday project - Julia

# Cascading failure in a small power grid

This exercise is based on the ["Cascading failure" example](https://juliadynamics.github.io/NetworkDynamics.jl/dev/generated/cascading_failure/) provided in the documentation of NetworkDynamics.jl.

In [2]:
# install needed packages in local environment
using Pkg
Pkg.activate(".")
Pkg.add("NetworkDynamics")
Pkg.add("Graphs")
Pkg.add("OrdinaryDiffEqTsit5")
Pkg.add("DiffEqCallbacks")
Pkg.add("Plots")
Pkg.add("SymbolicIndexingInterface")

  Activating new project at `~/Library/CloudStorage/OneDrive-UniversiteitUtrecht/Documents/work/teaching_/utrecht_summer_school/2026/sscs/projects/day3/julia_powergrid`
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed FunctionWrappersWrappers ─ v1.13.0
   Installed SteadyStateDiffEq ──────── v2.14.0
   Installed DiffEqBase ─────────────── v7.16.0
   Installed NNlib ──────────────────── v0.9.44
   Installed LineSearch ─────────────── v0.1.15
   Installed SciMLBase ──────────────── v3.49.1
   Installed SciMLOperators ─────────── v1.28.0
   Installed SpecialFunctions ───────── v2.9.0
   Installed NonlinearSolveBase ─────── v2.46.0
   Installed NonlinearSolveFirstOrder ─ v2.4.0
   Installed PrettyTables ───────────── v3.4.7
   Installed ArrayInterface ─────────── v7.30.0
   Installed LinearSolve ────────────── v5.12.0
   Installed NonlinearSolve ─────────── v4.27.0
   Installed UnsafeAtomics ──────────── v0.3.2
   Installed Recursive

In [4]:
# Load packages
using NetworkDynamics
using Graphs
using OrdinaryDiffEqTsit5
using DiffEqCallbacks
using Plots
import SymbolicIndexingInterface as SII

## Defining the Model

For the nodes we define the swing equation. State `v[1] = δ`, `v[2] = ω`.
The swing equation has three parameters: `p = (P_ref, I, γ)` where `P_ref`
is the power setpopint, `I` is the inertia and `γ` is the droop or damping coefficient.

The output of the node is just the first state. `g=1` is a shorthand for `g=StateMask(1:1)`
which implements a trivial output function `g` which just takes the first element of the state vector.

In [7]:
function swing_equation(dv, v, esum, p,t)
    P, I, γ = p
    dv[1] = v[2]
    dv[2] = P - γ * v[2] .+ esum[1]
    dv[2] = dv[2] / I
    nothing
end
vertex = VertexModel(f=swing_equation, g=1, sym=[:δ, :ω], psym=[:P_ref, :I=>1, :γ=>0.1])

VertexModel :VertexM PureStateMap()
 ├─ 2 states: [δ, ω]
 ├─ 1 output: [δ]
 └─ 3 params: [P_ref, I=1, γ=0.1]

Now we define the power flow along the edges (transmission lines), which depends on the sine of the phase differences and the coupling strength `K`. Also, we define a `limit` for the maximum value of the phase angle deviation (if this limit is exceeded, the line breaks down).

In [6]:
function simple_edge(e, v_s, v_d, (K,), t)
    e[1] = K * sin(v_s[1] - v_d[1])
end
edge = EdgeModel(;g=AntiSymmetric(simple_edge), outsym=:P, psym=[:K=>1.63, :limit=>1])

EdgeModel :StaticEdgeM PureFeedForward()
 ├─   0 states:  []  
 ├─ 1/1 outputs: src=[₋P] dst=[P]
 └─   2 params:  [K=1.63, limit=1]

Next, we define the network via a `SimpleGraph` constructed from the adjacency matrix.

In [8]:
A = [0 1 1 0 1;
    1 0 1 1 0;
    1 1 0 1 0;
    0 1 1 0 1;
    1 0 0 1 0]

g = SimpleGraph(A)
nw = Network(g, vertex, edge; dealias=true)

Network with 10 states and 29 parameters
 ├─ 5 vertices (1 unique type)
 └─ 7 edges (1 unique type)
Edge-Aggregation using SequentialAggregator(+)

Finally, we need to define the power input/output values at each node:

In [ ]:
# Power inputs/outputs
set_default!(nw, VIndex(1, :P_ref), -1.0) # load
set_default!(nw, VIndex(2, :P_ref),  1.5) # generator
set_default!(nw, VIndex(3, :P_ref), -1.0) # load
set_default!(nw, VIndex(4, :P_ref), -1.0) # load
set_default!(nw, VIndex(5, :P_ref),  1.5) # generator

Now that the power grid is set up, we can use the `find_fixpoint` function to find a valid initial condition of the network.
We also use `set_defaults!` to overwirte all the default values for states and parameters
with the one of the fixpoint, this means that we can allways re-extract this setpoint by
using `u0 = NWState(nw)`.


In [ ]:
u0 = find_fixpoint(nw)
set_defaults!(nw, u0)

We would now be ready to run a simulation, but since we are starting from a fixed point (stable frequency-synchronized state), nothing would change over time. 

We are interested in the effect of having a line failure at time `t=1`. This means that in the middle of the simulation, the system changes: the coupling strength `K` of the failing line suddenly drops to zero.

In NetworkDynamics.jl, this can be achieved using callbacks. The callback condition determines when the interference happens. The callback affect function specifies what will happen in that case.

We need two callbacks: one for tripping the first power line (here the 5th line) and one for other lines reaching their `limit`.

In [ ]:
cond = ComponentCondition([:P], [:limit]) do u, p, t
    abs(u[:P]) - p[:limit]
end
affect = ComponentAffect([], [:K]) do u, p, ctx
    println("Line $(ctx.eidx) tripped at t=$(ctx.integrator.t)")
    p[:K] = 0
end
edge_cb = ContinuousComponentCallback(cond, affect)

for i in 1:ne(g)
    edgemodel = nw[EIndex(i)]
    set_callback!(edgemodel, edge_cb)
end

trip_first_cb = PresetTimeComponentCallback(1.0, affect)
add_callback!(nw[EIndex(5)], trip_first_cb)

nw[EIndex(5)]

Now that we defined the callbacks and activated them for the `nw` model, we can run the simulation (by creating an `ODEProblem` and using the `solve` function with the `Tsit5` numerical solver for ODEs).

In [ ]:
u0 = NWState(nw)
time_interval = (0, 6)
prob = ODEProblem(nw, u0, time_interval)

sol = solve(prob, Tsit5());


Let's plot the result - did any other lines trip in a cascade?

In [ ]:
plot(sol; idxs=eidxs(sol, :, :P))

## Research questions

1. What is the role of the parameters `I` (inertia) and `gamma` (damping) in the swing equation, i.e. in the local dynamics of each node? Do you expect an increase in these parameters to increase or decrease the risk of cascades?

2. Which network structure is the most resilient against cascading failures?
    - Think about how you could design an experimental protocol to answer this question.
    - Set up a hypothesis for the answer to this question.
    - Test your hypothesis by re-running the code with different networks. 
    
### Constructing networks
Check out the documentation of `Graphs.jl`:
- [Scale-free](https://juliagraphs.org/Graphs.jl/dev/core_functions/simplegraphs_generators/#Graphs.SimpleGraphs.barabasi_albert-Tuple{Integer,%20Integer,%20Integer})
- [Small-world](https://juliagraphs.org/Graphs.jl/dev/core_functions/simplegraphs_generators/#Graphs.SimpleGraphs.watts_strogatz-Tuple{Integer,%20Integer,%20Real})
- [Configuration model](https://juliagraphs.org/Graphs.jl/dev/core_functions/simplegraphs_generators/#Graphs.SimpleGraphs.random_configuration_model-Union{Tuple{T},%20Tuple{Integer,%20Array{T}}}%20where%20T%3C:Integer)